# Hybrid Recommendation System using SVD

This notebook implements a hybrid recommendation system using Singular Value Decomposition (SVD), which combines collaborative filtering and content-based approaches for the educational content recommendation system.

## Objectives
1. Prepare user-item interaction matrix
2. Extract item features for content-based filtering
3. Implement collaborative filtering using SVD
4. Combine with content-based approach
5. Evaluate and tune the hybrid model

In [2]:
# Import required libraries
import os
import sys
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer

# Set up system path
sys.path.append(os.path.abspath(".."))

# For reproducibility
np.random.seed(42)

# Configure plotting aesthetics
sns.set_theme(style="whitegrid", palette="viridis")

## 1. Load and Prepare Data

In [3]:
# Load the datasets
lectures_data = pd.read_csv('../data/cleaned/cleaned_lectures.csv')
merged_data = pd.read_csv('../data/cleaned/merged_cleaned_data.csv')

# Display basic info about the datasets
print(f"Lectures dataset shape: {lectures_data.shape}")
print(f"Merged data shape: {merged_data.shape}")

Lectures dataset shape: (1021, 6)
Merged data shape: (117167, 14)


In [4]:
# Create subject categories based on ranges of tag values
# These mappings are based on TOEIC structure
def get_subject_category(tag):
    try:
        tag_val = float(tag)
        if 1 <= tag_val < 23:
            return "Listening Skills"
        elif 23 <= tag_val < 52:
            return "Reading Skills"
        elif 52 <= tag_val < 70:
            return "Speaking Skills"
        elif 70 <= tag_val < 150:
            return "Writing Skills"
        elif 150 <= tag_val < 200:
            return "Test Preparation"
        elif 200 <= tag_val < 300:
            return "Grammar & Vocabulary"
        else:
            return "General"
    except:
        return "General"

# Map part numbers to human-readable names
part_names = {
    0: "Introduction",
    1: "Listening Comprehension",
    2: "Reading Comprehension",
    3: "Grammar & Vocabulary",
    4: "Speaking Assessment",
    5: "Writing Exercises",
    6: "Practice Tests",
    7: "Additional Resources"
}

# Apply mappings
lectures_data['subject_category'] = lectures_data['tags'].apply(get_subject_category)
lectures_data['part_name'] = lectures_data['part'].map(part_names)

## 2. Create Bundle Features

Let's extract bundle information for content-based features in our hybrid model.

In [5]:
# Extract unique bundle information
bundle_info = merged_data.groupby('bundle_id').agg({
    'part': 'first',
    'tags': lambda x: ';'.join(set(str(i) for i in x if pd.notna(i))),
    'question_id': lambda x: len(set(x))  # Number of questions in bundle
}).reset_index()

# Rename columns for clarity
bundle_info.columns = ['bundle_id', 'part', 'tags', 'question_count']

# Map part to human-readable names
bundle_info['part_name'] = bundle_info['part'].map(part_names)

# Create subject category based on tags
bundle_info['subject_category'] = bundle_info['tags'].apply(get_subject_category)

# Calculate bundle popularity from interaction data
bundle_popularity = merged_data['bundle_id'].value_counts().reset_index()
bundle_popularity.columns = ['bundle_id', 'interaction_count']

# Calculate bundle difficulty from correct answer rates
bundle_difficulty = merged_data.groupby('bundle_id').apply(
    lambda x: (x['user_answer'] == x['correct_answer']).mean()
).reset_index()
bundle_difficulty.columns = ['bundle_id', 'success_rate']

# Merge all features
bundle_features = bundle_info.merge(bundle_popularity, on='bundle_id', how='left')
bundle_features = bundle_features.merge(bundle_difficulty, on='bundle_id', how='left')

# Fill missing values
bundle_features['interaction_count'] = bundle_features['interaction_count'].fillna(0)
bundle_features['success_rate'] = bundle_features['success_rate'].fillna(0.5)

print(f"Total unique bundles: {len(bundle_features)}")
bundle_features.head()

Total unique bundles: 8260


C:\Users\karat\AppData\Local\Temp\ipykernel_21004\3732587692.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bundle_difficulty = merged_data.groupby('bundle_id').apply(


,bundle_id,part,tags,question_count,part_name,subject_category,interaction_count,success_rate
0,b1,1,1;2;179;181,1,Listening Comprehension,General,6,0.833333
1,b10,1,17;7;182,1,Listening Comprehension,General,47,0.319149
2,b100,1,22;2;181,1,Listening Comprehension,General,7,1.000000
3,b1000,2,24;33;182;183,1,Reading Comprehension,General,46,0.760870
4,b1001,2,34;35;182;183,1,Reading Comprehension,General,12,0.583333


## 3. Create User-Item Interaction Matrix

In [6]:
# Filter users with minimum interactions for better model performance
min_interactions = 5
user_counts = merged_data['user_id'].value_counts()
valid_users = user_counts[user_counts >= min_interactions].index
filtered_data = merged_data[merged_data['user_id'].isin(valid_users)]

print(f"Original users: {merged_data['user_id'].nunique()}")
print(f"Filtered users with {min_interactions}+ interactions: {len(valid_users)}")
print(f"Retained {len(filtered_data) / len(merged_data):.2%} of interactions")

Original users: 999
Filtered users with 5+ interactions: 881
Retained 99.74% of interactions


In [7]:
# Create correctness as a measure of interaction quality
filtered_data = filtered_data.copy()  # Ensures modifications don't affect original data
filtered_data.loc[:, 'correct'] = (filtered_data['user_answer'] == filtered_data['correct_answer']).astype(int)

# Aggregate user-item interactions
user_item_data = filtered_data.groupby(['user_id', 'bundle_id']).agg(
    correctness_rate=('correct', 'mean'),
    interaction_count=('correct', 'count'),
    avg_time=('elapsed_time', 'mean'),
    part=('part', 'first'),
    tags=('tags', lambda x: ';'.join(set(str(i) for i in x)))
).reset_index()

# Create pivot table with interaction count as values
user_item_matrix = user_item_data.pivot_table(
    index='user_id', 
    columns='bundle_id', 
    values='interaction_count',
    fill_value=0
)

# Display user-item matrix characteristics
print(f"User-item matrix shape: {user_item_matrix.shape}")
print(f"Sparsity: {1 - (user_item_matrix > 0).sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.4f}")

User-item matrix shape: (881, 8256)
Sparsity: 0.9892
